# 04 Seasonal Trends

This notebook creates a **simple winter vs summer** trend analysis for all cities and exports a table for the Streamlit Notebook Insights page.

Season definition used:
- **Winter**: November to March (`11, 12, 1, 2, 3`)
- **Summer**: April to October (`4..10`)

In [1]:
import sys
import pandas as pd
import plotly.express as px

sys.path.insert(0, "..")
from utils import load_app_ready, export_df

In [2]:
df = load_app_ready()
df.shape

(1569542, 22)

In [3]:
required = ["city_name", "year", "month"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns for seasonal analysis: {missing}")

winter_months = {11, 12, 1, 2, 3}

seasonal = df.copy()
seasonal["season"] = seasonal["month"].apply(lambda m: "Winter" if int(m) in winter_months else "Summer")

agg = (
    seasonal.groupby(["city_name", "year", "season"], as_index=False)
    .agg(
        trips=("trip_id", "count"),
        avg_duration_minutes=("duration_seconds", lambda s: (s.dropna().mean() / 60) if len(s.dropna()) else None),
    )
)

agg["season"] = pd.Categorical(agg["season"], categories=["Winter", "Summer"], ordered=True)
agg = agg.sort_values(["city_name", "year", "season"]).reset_index(drop=True)
agg.head()

,city_name,year,season,trips,avg_duration_minutes
0,Bergen,2025,Winter,77578,9.299052
1,Bergen,2025,Summer,221986,11.307510
2,Oslo,2025,Winter,156172,10.652687
3,Oslo,2025,Summer,957132,12.959317
4,Trondheim,2025,Winter,9695,10.191496


In [4]:
fig = px.bar(
    agg,
    x="year",
    y="trips",
    color="season",
    barmode="group",
    facet_col="city_name",
    category_orders={"season": ["Winter", "Summer"]},
    title="Seasonal Trip Trends by City (Winter vs Summer)",
    labels={"trips": "Trips", "year": "Year", "city_name": "City"},
)
fig.update_layout(height=520, legend_title_text="Season")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

In [5]:
export_name = "seasonal_trends_city_year"
export_df(export_name, agg)
print(f"Exported: {export_name}.csv")

[bridge] Exported DataFrame: seasonal_trends_city_year.csv (target: NOTEBOOK_EXPORTS_PATH)
Exported: seasonal_trends_city_year.csv


## Notes

- Exported table path is resolved from the `.env` configuration (`NOTEBOOK_EXPORTS_PATH`).
- In Streamlit, open **Notebook Insights** and press **Reload exports** to view this output.
- This table is city-ready (`city_name` column) so it can be sliced by city in future dashboard components.